In [1]:
import torch
torch.__version__

'2.6.0.dev20241112+cu121'

In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
import torch
print(torch.version.cuda)

12.1


In [4]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.18.0
[]


In [5]:
!nvidia-smi

Wed Mar  5 13:01:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.07                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   57C    P8              9W /   95W |     120MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import torch
import os
import torchvision
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchinfo import summary

In [7]:
# EarlyStopping class
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        """
        Args:
        patience (int): How many epochs to wait after last time validation loss improved.
        delta (float): Minimum change in the monitored quantity to qualify as an improvement.
        """
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [8]:
# Set device and define model and optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [9]:
import torch
torch.cuda.is_available()

True

In [10]:
pretrained_vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT 
pretrained_vit = torchvision.models.vit_b_16(weights=pretrained_vit_weights).to(device)

In [11]:
# Freeze the base parameters
for parameter in pretrained_vit.parameters():
    parameter.requires_grad = False

In [12]:
# Change the classifier head
class_names = ['A','B','C','Five','Point','V']
pretrained_vit.heads = nn.Linear(in_features=768, out_features=len(class_names)).to(device)

In [13]:
# Define transforms and dataset
train_dir = r"D:\G\ISLAPHABETS\shp-marcel\shp_marcel_train\Marcel-Train"
test_dir = r"D:\G\ISLAPHABETS\shp-marcel\shp_marcel_test\Marcel-Test complex"

In [14]:
pretrained_vit_transforms = pretrained_vit_weights.transforms()

In [15]:
# Create dataloaders
def create_dataloaders(train_dir, test_dir, transform, batch_size, num_workers):
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)
    
    train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    return train_dataloader, test_dataloader, train_data.classes

train_dataloader_pretrained, test_dataloader_pretrained, class_names = create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=pretrained_vit_transforms,
    batch_size=32,
    num_workers=os.cpu_count()
)

In [16]:
# Set optimizer and loss function
optimizer = torch.optim.Adam(params=pretrained_vit.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

In [17]:
# Early stopping setup
early_stopping = EarlyStopping(patience=5, delta=0.01)

In [18]:
import time

# Track total training time
def train_with_early_stopping(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs, device, early_stopping):
    best_val_loss = float('inf')
    
    # Record the start time
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Train accuracy
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        # Validation step
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for images, labels in test_dataloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = loss_fn(outputs, labels)
                val_loss += loss.item()
                
                # Validation accuracy
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        avg_train_loss = running_loss / len(train_dataloader)
        avg_val_loss = val_loss / len(test_dataloader)
        train_accuracy = 100 * correct_train / total_train
        val_accuracy = 100 * correct_val / total_val

        print(f"Epoch [{epoch+1}/{epochs}], "
              f"Train Loss: {avg_train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, "
              f"Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

        # Early stopping check
        early_stopping(avg_val_loss)

        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    # Record the end time
    end_time = time.time()

    # Calculate and print total training time
    total_training_time = end_time - start_time
    print(f"Total Training Time: {total_training_time/60:.2f} minutes")


In [19]:
# Train with early stopping
train_with_early_stopping(pretrained_vit, 
                          train_dataloader_pretrained, 
                          test_dataloader_pretrained, 
                          optimizer, 
                          loss_fn, 
                          epochs=50, 
                          device=device,
                          early_stopping=early_stopping)

Epoch [1/50], Train Loss: 0.6420, Train Accuracy: 81.51%, Val Loss: 1.2268, Val Accuracy: 52.71%
Epoch [2/50], Train Loss: 0.2348, Train Accuracy: 95.26%, Val Loss: 1.0960, Val Accuracy: 61.37%
Epoch [3/50], Train Loss: 0.1547, Train Accuracy: 97.39%, Val Loss: 1.0817, Val Accuracy: 61.37%
Epoch [4/50], Train Loss: 0.1156, Train Accuracy: 98.26%, Val Loss: 1.0735, Val Accuracy: 60.65%
Epoch [5/50], Train Loss: 0.0936, Train Accuracy: 98.56%, Val Loss: 1.0634, Val Accuracy: 63.90%
Epoch [6/50], Train Loss: 0.0770, Train Accuracy: 98.95%, Val Loss: 1.0719, Val Accuracy: 64.62%
Epoch [7/50], Train Loss: 0.0663, Train Accuracy: 99.14%, Val Loss: 1.0871, Val Accuracy: 64.98%
Epoch [8/50], Train Loss: 0.0575, Train Accuracy: 99.34%, Val Loss: 1.1058, Val Accuracy: 63.90%
Epoch [9/50], Train Loss: 0.0500, Train Accuracy: 99.43%, Val Loss: 1.1078, Val Accuracy: 62.82%
Epoch [10/50], Train Loss: 0.0458, Train Accuracy: 99.32%, Val Loss: 1.1429, Val Accuracy: 63.54%
Early stopping triggered!
Tot

# TESTING

In [29]:
import os
import cv2
import torch
import torch.nn.functional as F
import time
import numpy as np
from torchvision import transforms
from PIL import Image
from sklearn.metrics import classification_report

# Define the path to the folder containing test images
test_images_folder = r"D:\G\ISLAPHABETS\shp-marcel\shp_marcel_test\10"

# Ensure the folder path exists
if not os.path.exists(test_images_folder):
    print(f"Error: The folder '{test_images_folder}' does not exist.")
else:
    print(f"Testing images from folder: {test_images_folder}")

# Define the same transformation used during training
real_time_transform = pretrained_vit_weights.transforms()

# Function to test model on images and calculate evaluation metrics
def test_on_images(model, image_folder, device, class_names):
    model.eval()  # Set model to evaluation mode
    
    class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

    correct_predictions = 0
    total_images = 0
    total_inference_time = 0.0

    y_true = []  # True labels
    y_pred = []  # Predicted labels

    # Loop through each class folder
    for class_folder in os.listdir(image_folder):
        class_path = os.path.join(image_folder, class_folder)

        if not os.path.isdir(class_path):
            continue  # Skip non-folder files

        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        print(f"Testing {len(image_files)} images from class '{class_folder}'.")

        for image_file in image_files:
            image_path = os.path.join(class_path, image_file)
            
            try:
                # Load and transform image
                image = Image.open(image_path).convert("RGB")
                transformed_image = real_time_transform(image).unsqueeze(0).to(device)

                # Measure inference time
                start_time = time.time()
                with torch.no_grad():
                    outputs = model(transformed_image)
                    probabilities = F.softmax(outputs, dim=1)  # Convert logits to probabilities
                    confidence, predicted = torch.max(probabilities, 1)  # Get highest probability and class index
                end_time = time.time()

                inference_time = end_time - start_time
                total_inference_time += inference_time

                predicted_class = class_names[predicted.item()]
                confidence_score = confidence.item() * 100  # Convert to percentage

                # Store true and predicted labels
                if class_folder in class_to_idx:
                    y_true.append(class_to_idx[class_folder])  # True label
                    y_pred.append(predicted.item())  # Predicted label

                # Compare prediction with actual class
                actual_class = class_folder
                is_correct = (predicted_class == actual_class)

                if is_correct:
                    correct_predictions += 1
                
                total_images += 1

                # Print result with confidence score and inference time
                print(f"Image: {image_file} --> Predicted: {predicted_class} ({confidence_score:.2f}%), "
                      f"Actual: {actual_class}, Correct: {is_correct}, Inference Time: {inference_time:.4f} sec")

                # Display image with prediction, confidence score, and inference time
                img = cv2.imread(image_path)
                img = cv2.resize(img, (400, 400))  # Resize for better visualization

                label_text = f"{predicted_class} ({confidence_score:.1f}%)"
                cv2.putText(img, label_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(img, f"{inference_time:.3f}s", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

                cv2.imshow("Prediction", img)
                cv2.waitKey(500)  # Display each image for 0.5 seconds

            except Exception as e:
                print(f"Error processing image {image_file}: {e}")

    cv2.destroyAllWindows()

    # Compute accuracy
    if total_images > 0:
        accuracy = (correct_predictions / total_images) * 100
        avg_inference_time = total_inference_time / total_images
        print(f"\nTest Accuracy: {accuracy:.2f}% ({correct_predictions}/{total_images} correct)")
        print(f"Average Inference Time per Image: {avg_inference_time:.4f} sec")

        # Compute Precision, Recall, and F1-score
        print("\nClassification Report:\n")
        print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    else:
        print("\nNo images found for testing.")

# Run testing on folder images
test_on_images(pretrained_vit, test_images_folder, device, class_names)


Testing images from folder: D:\G\ISLAPHABETS\shp-marcel\shp_marcel_test\10
Testing 10 images from class 'A'.
Image: A-uniform01.jpg --> Predicted: A (81.25%), Actual: A, Correct: True, Inference Time: 0.0101 sec
Image: A-uniform02.jpg --> Predicted: A (92.91%), Actual: A, Correct: True, Inference Time: 0.0115 sec
Image: A-uniform08.jpg --> Predicted: A (99.98%), Actual: A, Correct: True, Inference Time: 0.0078 sec
Image: A-uniform09.jpg --> Predicted: A (99.68%), Actual: A, Correct: True, Inference Time: 0.0082 sec
Image: A-uniform10.jpg --> Predicted: A (95.49%), Actual: A, Correct: True, Inference Time: 0.0050 sec
Image: A-uniform11.jpg --> Predicted: A (95.01%), Actual: A, Correct: True, Inference Time: 0.0075 sec
Image: A-uniform13.jpg --> Predicted: A (99.63%), Actual: A, Correct: True, Inference Time: 0.0078 sec
Image: A-uniform14.jpg --> Predicted: A (98.88%), Actual: A, Correct: True, Inference Time: 0.0080 sec
Image: A-uniform21.jpg --> Predicted: A (77.95%), Actual: A, Correc